In [60]:
from google.colab import files
import pandas as pd
import io
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report

In [62]:
upload = files.upload()

Saving battery_synthetic_dataset.csv to battery_synthetic_dataset (1).csv


In [63]:
df = pd.read_csv(io.BytesIO(upload['battery_synthetic_dataset (1).csv']))

In [64]:
data = df.copy()

In [66]:
x = data.drop(["label"],axis=1)
y = data[["label"]]

In [48]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [49]:
scaler = StandardScaler()
x_train_s = scaler.fit_transform(x_train)
x_test_s = scaler.transform(x_test)

In [50]:
def sigmoid(x):
  return 1/(1+np.exp(-x))

def sigmoid_derivative(x):
  return x*(1-x)

def relu(x):
  return np.maximum(0,x)

def relu_derivative(x):
  return (x>0).astype(float)

In [67]:
input_size = x_train_s.shape[1]
hidden_size = 10
output_size = y_train.shape[1]

learning_rate = 1e-3
epochs = 10000

In [68]:
np.random.seed(42)
weights_input_hidden = np.random.uniform(-1, 1, size=(input_size, hidden_size))
weights_hidden_output = np.random.uniform(-1, 1, size=(hidden_size, output_size))
bias_input_hidden = np.zeros((1, hidden_size))
bias_hidden_output = np.zeros((1, output_size))

In [69]:
y_train_arr = y_train.values.reshape(-1, 1)

In [70]:
for epoch in range(epochs):
    # forward pass (training data, scaled)
    hidden_layer_input = np.dot(x_train_s, weights_input_hidden) + bias_input_hidden
    hidden_layer_output = sigmoid(hidden_layer_input)
    output_layer_input = np.dot(hidden_layer_output, weights_hidden_output) + bias_hidden_output
    predicted_output = sigmoid(output_layer_input)

    # backward pass
    error = y_train_arr - predicted_output
    d_predicted_output = error * sigmoid_derivative(predicted_output)
    error_hidden_layer = d_predicted_output.dot(weights_hidden_output.T)
    d_hidden_layer = error_hidden_layer * sigmoid_derivative(hidden_layer_output)

    grad_w2 = hidden_layer_output.T.dot(d_predicted_output)
    grad_w1 = x_train_s.T.dot(d_hidden_layer)
    grad_b2 = np.sum(d_predicted_output, axis=0, keepdims=True)
    grad_b1 = np.sum(d_hidden_layer, axis=0, keepdims=True)

    weights_hidden_output += learning_rate * grad_w2
    weights_input_hidden += learning_rate * grad_w1
    bias_hidden_output += learning_rate * grad_b2
    bias_input_hidden += learning_rate * grad_b1

    if (epoch + 1) % 1000 == 0 or epoch == 0:
        mse = np.mean(np.square(y_train_arr - predicted_output))
        print(f"Epoch {epoch+1}/{epochs}, Train MSE: {mse:.5f}")

Epoch 1/10000, Train MSE: 0.43018
Epoch 1000/10000, Train MSE: 0.06631
Epoch 2000/10000, Train MSE: 0.06564
Epoch 3000/10000, Train MSE: 0.06549
Epoch 4000/10000, Train MSE: 0.06539
Epoch 5000/10000, Train MSE: 0.06530
Epoch 6000/10000, Train MSE: 0.06522
Epoch 7000/10000, Train MSE: 0.06515
Epoch 8000/10000, Train MSE: 0.06508
Epoch 9000/10000, Train MSE: 0.06501
Epoch 10000/10000, Train MSE: 0.06494


In [71]:
hidden_layer_input = np.dot(x_test_s, weights_input_hidden) + bias_input_hidden
hidden_layer_output = sigmoid(hidden_layer_input)
output_layer_input = np.dot(hidden_layer_output, weights_hidden_output) + bias_hidden_output
predicted_output = sigmoid(output_layer_input)

In [73]:
print(f"\nR2 score: {r2_score(y_test, predicted_output)}")
print(f"MSE: {mean_squared_error(y_test, predicted_output)}")

predicted_labels = (predicted_output > 0.5).astype(int)
print(f"\nAccuracy: {accuracy_score(y_test, predicted_labels)}")
print(classification_report(y_test, predicted_labels))


R2 score: 0.7377545883204454
MSE: 0.06078193029202875

Accuracy: 0.905
              precision    recall  f1-score   support

           0       0.84      0.92      0.88        73
           1       0.95      0.90      0.92       127

    accuracy                           0.91       200
   macro avg       0.89      0.91      0.90       200
weighted avg       0.91      0.91      0.91       200

